# Google Colab Integration Demo

This notebook demonstrates the Google Colab integration utilities for the RLHF Phi-3 pipeline.

## Features Demonstrated:
- Session management and timeout monitoring
- Google Drive integration for checkpoint persistence
- Memory optimization and monitoring
- Progress tracking with user-friendly interfaces
- Error handling with troubleshooting guides
- Session resume capabilities

In [ ]:
# Install required dependencies (run this first in Colab)
!pip install transformers torch peft trl datasets accelerate wandb psutil GPUtil

# Import the Colab utilities
from colab_utils import (
    setup_colab_session,
    monitor_session_health,
    handle_training_error,
    emergency_checkpoint_save,
    create_session_resume_guide
)

## 1. Setup Colab Session

Initialize the complete Colab environment with session management, Google Drive integration, and progress tracking.

In [ ]:
# Setup the complete Colab session
session_manager, drive_manager, progress_tracker = setup_colab_session(
    config_path="/content/drive/MyDrive/rlhf-phi3/config.yaml",
    enable_timeout_monitoring=True
)

print("✅ Colab session setup completed!")
print(f"📊 Session timeout: {session_manager.session_timeout_hours} hours")
print(f"💾 Drive mounted: {drive_manager.is_mounted()}")
print(f"📈 Progress tracker ready with {len(progress_tracker.error_messages)} error guides")

## 2. Monitor Session Health

Check current session status, memory usage, and time remaining.

In [ ]:
# Get comprehensive session health information
health_info = monitor_session_health(display_warnings=True)

print("📊 Session Health Report:")
print(f"⏱️  Elapsed time: {health_info['session_info']['elapsed_hours']:.1f} hours")
print(f"⏰ Remaining time: {health_info['time_recommendations']['remaining_hours']:.1f} hours")

# Memory usage
memory = health_info['session_info']['memory_usage']
if 'gpu_0' in memory['gpu_memory']:
    gpu_mem = memory['gpu_memory']['gpu_0']
    print(f"🎮 GPU Memory: {gpu_mem['percent_used']:.1f}% used ({gpu_mem['name']})")

sys_mem = memory['system_memory']
print(f"💻 System Memory: {sys_mem['percent_used']:.1f}% used")

# Recommendations
if health_info['time_recommendations']['should_checkpoint']:
    print("⚠️  Recommendation: Save checkpoints soon!")
if health_info['memory_pressure']['critical_level']:
    print("🚨 Warning: High memory usage detected!")

## 3. Progress Tracking Demo

Demonstrate progress tracking for long-running operations.

In [ ]:
import time

# Example: Track a training operation
with progress_tracker.track_operation("SFT Training", total_steps=100) as operation_id:
    for step in range(1, 101):
        # Simulate training step
        time.sleep(0.05)  # 50ms per step
        
        # Update progress every 10 steps
        if step % 10 == 0:
            progress_tracker.update_progress(
                operation_id, 
                step, 
                f"Loss: {3.5 - (step * 0.02):.3f}"
            )
            
            # Display progress bar
            progress_bar = progress_tracker.display_progress_bar(operation_id)
            print(progress_bar)

print("✅ Training simulation completed!")

## 4. Memory Optimization

Demonstrate memory optimization and monitoring.

In [ ]:
# Check current memory usage
initial_memory = session_manager.get_memory_usage()
print("📊 Memory Usage Before Optimization:")
if 'gpu_0' in initial_memory['gpu_memory']:
    gpu_mem = initial_memory['gpu_memory']['gpu_0']
    print(f"🎮 GPU: {gpu_mem['reserved_gb']:.1f}GB reserved, {gpu_mem['percent_used']:.1f}% used")

sys_mem = initial_memory['system_memory']
print(f"💻 System: {sys_mem['used_gb']:.1f}GB used, {sys_mem['percent_used']:.1f}% used")

# Perform memory optimization
print("\n🔧 Optimizing memory...")
optimization_result = session_manager.optimize_memory()

print("\n📈 Memory Optimization Results:")
if 'gpu_gb' in optimization_result['memory_freed']:
    print(f"🎮 GPU memory freed: {optimization_result['memory_freed']['gpu_gb']:.2f}GB")
print(f"💻 System memory freed: {optimization_result['memory_freed']['system_gb']:.2f}GB")

## 5. Error Handling Demo

Demonstrate error handling with user-friendly guidance.

In [ ]:
# Simulate different types of training errors
error_examples = [
    Exception("CUDA out of memory. Tried to allocate 2.00 GiB"),
    Exception("Session timeout occurred after 12 hours"),
    Exception("Failed to mount Google Drive"),
    Exception("Dataset 'nonexistent-dataset' not found"),
    Exception("Training loss is NaN at step 150")
]

for i, error in enumerate(error_examples, 1):
    print(f"\n{'='*60}")
    print(f"Error Example {i}: {str(error)}")
    print(f"{'='*60}")
    
    error_info = handle_training_error(error, f"During training step {i*100}")
    
    print(f"🏷️  Error Type: {error_info['error_type']}")
    
    if error_info['guidance']:
        print(f"💡 Guidance: {error_info['guidance']['message']}")
        print("🔧 Troubleshooting Steps:")
        for j, step in enumerate(error_info['guidance']['troubleshooting'][:3], 1):
            print(f"   {j}. {step}")
    else:
        print("❓ No specific guidance available for this error type")

## 6. Session State Persistence

Demonstrate saving and loading session state for resumption.

In [ ]:
# Simulate training state
training_state = {
    "current_stage": "sft",
    "current_epoch": 2,
    "current_step": 450,
    "best_loss": 1.234,
    "model_checkpoint": "/content/drive/MyDrive/rlhf-phi3/checkpoints/sft_epoch_2_step_450.pt",
    "optimizer_state": "/content/drive/MyDrive/rlhf-phi3/checkpoints/optimizer_sft_450.pt"
}

print("💾 Saving session state...")
saved_path = session_manager.save_session_state(training_state, drive_manager)
print(f"✅ Session state saved to: {saved_path}")

print("\n📂 Loading session state...")
loaded_state = session_manager.load_session_state(drive_manager)

if loaded_state:
    print("✅ Session state loaded successfully!")
    print(f"📊 Training State:")
    state_data = loaded_state['state_data']
    print(f"   Stage: {state_data['current_stage']}")
    print(f"   Epoch: {state_data['current_epoch']}")
    print(f"   Step: {state_data['current_step']}")
    print(f"   Best Loss: {state_data['best_loss']}")
else:
    print("❌ Failed to load session state")

## 7. Resume Instructions Generation

Generate detailed instructions for resuming training after session timeout.

In [ ]:
# Generate resume instructions
checkpoint_path = "/content/drive/MyDrive/rlhf-phi3/checkpoints/sft_epoch_2_step_450.pt"
stage = "sft"
step = 450

print("📝 Generating session resume instructions...")
instructions = create_session_resume_guide(
    checkpoint_path=checkpoint_path,
    stage=stage,
    step=step,
    save_to_drive=True
)

print("✅ Resume instructions generated and saved!")
print("\n📋 Preview of resume instructions:")
print(instructions[:500] + "..." if len(instructions) > 500 else instructions)

## 8. Emergency Checkpoint Save

Demonstrate emergency checkpoint saving for session timeout scenarios.

In [ ]:
# This would be used with actual model and optimizer objects
# For demo purposes, we'll show the interface

print("🚨 Emergency Checkpoint Save Demo")
print("\nIn a real scenario, you would use:")
print("""
# When session timeout is imminent
try:
    checkpoint_path = emergency_checkpoint_save(
        checkpoint_manager=checkpoint_manager,
        model=model,
        optimizer=optimizer,
        step=current_step,
        stage=current_stage
    )
    print(f"Emergency checkpoint saved: {checkpoint_path}")
    
    # Generate resume instructions
    create_session_resume_guide(checkpoint_path, current_stage, current_step)
    
except Exception as e:
    print(f"Emergency save failed: {e}")
    # Manual backup procedures...
""")

print("\n💡 The emergency save function automatically:")
print("   - Saves model and optimizer state")
print("   - Marks checkpoint as emergency save")
print("   - Provides immediate feedback")
print("   - Generates resume instructions")

## Summary

This demo showcased the comprehensive Google Colab integration features:

✅ **Session Management**: Automatic timeout monitoring and health checks  
✅ **Google Drive Integration**: Seamless checkpoint persistence across sessions  
✅ **Memory Optimization**: Automatic memory cleanup and monitoring  
✅ **Progress Tracking**: User-friendly progress bars and status updates  
✅ **Error Handling**: Intelligent error classification with troubleshooting guides  
✅ **Session Resume**: Detailed instructions for continuing after interruptions  
✅ **Emergency Features**: Quick checkpoint saves for timeout scenarios  

These utilities ensure a smooth training experience within Google Colab's constraints while providing robust error recovery and user guidance.